# **Fine-tuning mBART50 for En-Vi Machine Translation**

In [ ]:
!pip install -q transformers sentencepiece datasets accelerate evaluate sacrebleu

## **Dataset**

In [ ]:
from datasets import load_dataset

ds = load_dataset("thainq107/iwslt2015-en-vi")

In [ ]:
ds

In [ ]:

#train_50 = ds['train'].train_test_split(test_size=0.5, seed=42)['train']
#val_70 = ds['validation'].train_test_split(test_size=0.3, seed=42)['train']

#print("Train (50%):", len(train_50))  
#print("Val (70%):", len(val_70))      

#from datasets import DatasetDict
#ds_scaled = DatasetDict({
    #'train': train_50,
    #'validation': val_70
#})

## **Tokenizer**

In [ ]:
from transformers import AutoTokenizer

model_name = "facebook/mbart-large-50-many-to-many-mmt"
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
len(tokenizer)

## **Encoding**

In [ ]:
import torch

MAX_LEN = 50

def preprocess_function(examples):
    input_ids_list = []
    labels_list = []
    
    en_sentences = examples["en"]
    vi_sentences = examples["vi"]
    
    for i in range(len(en_sentences)):
        en_text = en_sentences[i]
        vi_text = vi_sentences[i]
        
        if i % 2 == 0:
            src_ids = tokenizer(en_text, padding='max_length', truncation=True, max_length=MAX_LEN)['input_ids']
            tgt_ids = tokenizer(text_target=vi_text, padding='max_length', truncation=True, max_length=MAX_LEN)['input_ids']
        else:
            src_ids = tokenizer(vi_text, padding='max_length', truncation=True, max_length=MAX_LEN)['input_ids']
            tgt_ids = tokenizer(text_target=en_text, padding='max_length', truncation=True, max_length=MAX_LEN)['input_ids']
            
        input_ids_list.append(src_ids)
        labels_list.append(tgt_ids)

    processed_labels = [
        [-100 if item == tokenizer.pad_token_id else item for item in label]
        for label in labels_list
    ]
    return {
        "input_ids": torch.tensor(input_ids_list),
        "labels": torch.tensor(processed_labels)
    }

preprocessed_ds = ds.map(preprocess_function, batched=True)

In [ ]:
preprocessed_ds['train'][0]

## **Model**

In [ ]:
from transformers import AutoModelForSeq2SeqLM

model_name = "facebook/mbart-large-50-many-to-many-mmt"
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

In [ ]:
model

## **Evaluate**

In [ ]:
import numpy as np
import evaluate
metric = evaluate.load("sacrebleu")

def postprocess_text(preds, labels):
    preds = [pred.strip() for pred in preds]
    labels = [[label.strip()] for label in labels]

    return preds, labels

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]

    preds= np.where(preds != -100, preds, tokenizer.pad_token_id)
    decoded_preds = tokenizer.batch_decode(
        preds, skip_special_tokens=True, clean_up_tokenization_spaces=True
        )

    labels= np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(
        labels, skip_special_tokens=True, clean_up_tokenization_spaces=True
        )

    decoded_preds, decoded_labels = postprocess_text(
        decoded_preds, decoded_labels
    )

    result = metric.compute(predictions=decoded_preds, references=decoded_labels)
    result = {"bleu": result["score"]}

    return result

## **Trainer**

In [ ]:
# Disable wandb
import os
os.environ['WANDB_DISABLED'] = 'true'
#os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'
# # Use wandb
# import wandb
# wandb.init(
#     project="en-vi-machine-translation",
#     name="mbart50" #
# )

In [ ]:
!pip install -U bitsandbytes scipy

In [ ]:
from transformers import Seq2SeqTrainingArguments, DataCollatorForSeq2Seq, Seq2SeqTrainer

training_args = Seq2SeqTrainingArguments(
    output_dir="./en-vi-bidirectional-mbart50",
    logging_dir="logs",
    fp16=True,
    gradient_checkpointing=True,
    optim="adamw_bnb_8bit",
    logging_steps=1000,
    predict_with_generate=True,
    eval_strategy="steps",
    eval_steps=1000,
    save_strategy="steps",
    save_steps=1000,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    save_total_limit=1,
    num_train_epochs=3,
    load_best_model_at_end=True,
    # report_to="wandb"
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer,
    model=model
)

trainer = Seq2SeqTrainer(
    model,
    training_args,
    train_dataset=preprocessed_ds['train'],
    eval_dataset=preprocessed_ds['validation'],
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

In [ ]:
import gc

gc.collect()
torch.cuda.empty_cache()


In [ ]:
trainer.train()

In [ ]:
trainer.push_to_hub(token="...")

## **Inference**

In [ ]:
model_name = "Thehien2k7/en-vi-bidirectional-mbart50"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

### **Greedy Search**

In [ ]:
src_text = "I go to school"
encoded_text = tokenizer(src_text, return_tensors="pt")
generated_tokens = model.generate(
    **encoded_text
)
tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)

### **Beam search**

In [ ]:
src_text = "In the next step, we consider the next possible tokens for each of the three branches we created in the previous step."
encoded_text = tokenizer(src_text, return_tensors="pt")
generated_tokens = model.generate(
    **encoded_text,
    num_beams=5,
)
tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)